In [1]:
# ================================
# Standard library
# ================================
import os
from datetime import date, timedelta

# ================================
# Third-party libraries
# ================================
import pandas as pd
import requests
from dotenv import load_dotenv
from mistralai.client import Mistral

# ================================
# LangChain core
# ================================
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

# ================================
# LangChain community
# ================================
from langchain_community.vectorstores import FAISS

# ================================
# LangChain Mistral
# ================================
from langchain_mistralai import ChatMistralAI, MistralAIEmbeddings

In [2]:
load_dotenv()
openagenda_api_key = os.getenv("OPENAGENDA_API_KEY")

headers = {"key": openagenda_api_key}
url = "https://api.openagenda.com/v2/agendas"

response = requests.get(url, headers=headers, timeout=30)
response.raise_for_status()

data = response.json()
print(data)

{'after': [1, 530050], 'agendas': [{'uid': 41648, 'image': 'https://cdn.openagenda.com/main/agenda41648.4b9a3a0a2a41480f879368e72a643205.jpg?__ts=1765186692460', 'description': "Du 14 au 21 mars 2026, tous ensemble pour l'éveil du tout-petit ! \nUne semaine d'ateliers & activités à travers toute la France, pour réunir le trio parent-enfant-professionnel.", 'official': True, 'title': 'La Semaine Nationale de la Petite Enfance', 'slug': 'semainepetiteenfance'}, {'uid': 50100, 'image': 'https://cdn.openagenda.com/main/agenda50100.43b6afe0d5804f8b93019e349c254b42.jpg?__ts=1761124850322', 'description': "« La classe, l’œuvre ! » est une opération qui offre à des classes la possibilité de construire un projet d'éducation artistique et culturelle et de le restituer durant La Nuit européenne des musées, le 23 mai 2026. Retrouvez les informations sur notre site Internet", 'official': True, 'title': "La classe, l'œuvre ! 2026", 'slug': 'lclo-2026'}, {'uid': 62695, 'image': 'https://cdn.openagend

In [3]:
# Fenêtre temporelle : événements de moins d’un an (consigne)
today = date.today()
date_from = (today - timedelta(days=365)).isoformat()
date_to = today.isoformat()

print(date_from)
print(date_to)

2025-03-23
2026-03-23


In [4]:
Zone_choisie = "Montpellier"
scope_choisi = "city"

In [5]:
#pour test uniquement
def search_event(query: str = None, limit: int = 10):
    
    url = f"https://api.openagenda.com/v2/agendas"

    params = {"size": limit}
    if query : params["includeFields"] = query

    #get : lire les données
    response = requests.get(
        url,
        headers=headers,
        params=params,
        timeout=30
    )

    response.raise_for_status()

    return response.json()

<h3>Schéma type d'un document</h3>
<div style="
  border-left:5px solid #48C9B0;
  background-color:#f8fdfc;
  color:#000000;
  padding:14px 18px;
  margin:18px 0
">
<strong>Objectif :</strong><br>
</div>

<ul style="margin-left:18px; line-height:1.7">
<li></li>
<li></li>
<li></li>
</ul>



In [6]:
def _safe(value):
    return "" if value is None else str(value)

def build_event_document(event: dict) -> Document:
    event_uid = _safe(event.get("event_uid"))
    title = _safe(event.get("title"))
    description = _safe(event.get("description"))
    location_name = _safe(event.get("location_name"))
    city = _safe(event.get("city"))
    region = _safe(event.get("region"))
    first_date = _safe(event.get("first_date"))
    last_date = _safe(event.get("last_date"))
    event_type = _safe(event.get("event_type"))
    source_url = _safe(event.get("source_url"))
    agenda_uid = _safe(event.get("agenda_uid"))

    parts = [
        f"Titre : {title}",
        f"Description : {description}",
    ]

    if location_name:
        parts.append(f"Lieu : {location_name}")
    if city:
        parts.append(f"Ville : {city}")
    if region:
        parts.append(f"Région : {region}")
    if first_date:
        parts.append(f"Date de début : {first_date}")
    if last_date:
        parts.append(f"Date de fin : {last_date}")
    if event_type:
        parts.append(f"Type d'événement : {event_type}")

    page_content = "\n".join(parts)

    metadata = {
        "doc_id": f"openagenda_{event_uid}",
        "chunk_id": f"openagenda_{event_uid}_0",
        "source": "openagenda",
        "event_uid": event_uid,
        "agenda_uid": agenda_uid,
        "title": title,
        "city": city,
        "region": region,
        "location_name": location_name,
        "first_date": first_date,
        "last_date": last_date,
        "event_type": event_type,
        "url": source_url,
        "lang": "fr"
    }

    return Document(page_content=page_content, metadata=metadata)

In [7]:
def search_agendas_for_zone(zone: str, size: int = 10, official: bool = True) -> list[dict]:
    if not openagenda_api_key:
        raise ValueError("OPENAGENDA_API_KEY manquante dans le fichier .env")

    url = "https://api.openagenda.com/v2/agendas"
    headers = {"key": openagenda_api_key}
    params = {
        "search": zone,
        "size": size,
    }
    if official: params["official"] = 1 #on va eviter les agendas amateurs 

    # envoie de la requete api 
    response = requests.get(url, headers=headers, params=params, timeout=30)
    response.raise_for_status()
    payload = response.json()

    if "agendas" not in payload:
        raise ValueError(f"Réponse inattendue : {list(payload.keys())}")

    return payload["agendas"]

<h3>récuperation des evenements OpenAgenda</h3>
<div style="
  border-left:5px solid #48C9B0;
  background-color:#f8fdfc;
  color:#000000;
  padding:14px 18px;
  margin:18px 0
">
<strong>Objectif :</strong><br>
</div>

<ul style="margin-left:18px; line-height:1.7">
<li></li>
<li></li>
<li></li>
</ul>


1. Récupérer les événements filtrés

Commence par une fonction qui interroge OpenAgenda avec pagination.

In [8]:
def fetch_openagenda_events(
    agenda_uid: str,
    zone: str,
    scope: str = "city",
    limit: int = 300,
    max_pages: int = 20,
) -> list[dict]:

    if not openagenda_api_key:
        raise ValueError("OPENAGENDA_API_KEY manquante dans le fichier .env")

    all_events = []
    offset = 0

    for _ in range(max_pages):
        url = f"https://openagenda.com/agendas/{agenda_uid}/events.json"

        params = {
            "key": openagenda_api_key,
            "limit": limit,
            "offset": offset,
            "oaq[passed]": 1,
            "oaq[from]": date_from,
            "oaq[to]": date_to,
            "oaq[what]": zone,
            "oaq[scope]": scope,
            "oaq[order]": "latest"
        }

        response = requests.get(url, params=params, timeout=30)
        response.raise_for_status()
        payload = response.json()

        if "events" not in payload:
            raise ValueError(
                f"Réponse inattendue de l'API : {list(payload.keys())}"
            )

        events = payload["events"]

        if not events:
            break

        all_events.extend(events)

        total = payload.get("total", 0)
        offset += limit

        if offset >= total:
            break

    return all_events

2. Normaliser dans un DataFrame

L’API renvoie un JSON riche. Pour travailler proprement, tu l’aplatis avec pd.json_normalize().

In [9]:
#transforme json de reponse en panda dataset conforme à build_event
def normalize_events(events: list[dict]) -> pd.DataFrame:
    #schema de build_event_document().

    schema_cols = [
        "event_uid",
        "agenda_uid",
        "title",
        "description",
        "location_name",
        "city",
        "region",
        "first_date",
        "last_date",
        "event_type",
        "source_url"
    ]

    if not events:
        return pd.DataFrame(columns=schema_cols)

    df = pd.json_normalize(events)

    rename_map = {
        "uid": "event_uid",
        "agenda.uid": "agenda_uid",
        "title.fr": "title",
        "description.fr": "description",
        "canonicalUrl": "source_url",
        "firstDate": "first_date",
        "lastDate": "last_date",
        "type": "event_type",
        "eventType": "event_type",
        "location.name": "location_name",
        "location.city": "city",
        "location.region": "region"
    }

    existing_cols = [c for c in rename_map if c in df.columns]
    df = df[existing_cols].copy()
    df = df.rename(columns={c: rename_map[c] for c in existing_cols})

    for col in schema_cols:
        if col not in df.columns:
            df[col] = ""

    for col in schema_cols:
        df[col] = df[col].fillna("").astype(str).str.strip()

    df["title"] = df["title"].str.replace(r"\s+", " ", regex=True)
    df["description"] = df["description"].str.replace(r"\s+", " ", regex=True)

    return df[schema_cols].copy()

3. Construire le texte pour l’indexation

le format document = [titre, date, informations importantes, metadata].

In [10]:
def build_indexable_text(df: pd.DataFrame) -> pd.DataFrame:
    #recap pour embedding
    df = df.copy()

    df["text_for_embedding"] = (
        "Titre : " + df["title"].fillna("") + "\n"
        + "Description : " + df["description"].fillna("") + "\n"
        + "Lieu : " + df["location_name"].fillna("") + "\n"
        + "Ville : " + df["city"].fillna("") + "\n"
        + "Région : " + df["region"].fillna("") + "\n"
        + "Date de début : " + df["first_date"].fillna("").astype(str) + "\n"
        + "Date de fin : " + df["last_date"].fillna("").astype(str) + "\n"
        + "Type d'événement : " + df["event_type"].fillna("")
    )

    return df

In [11]:
#recherche des evenmements de montpellier
agendas = search_agendas_for_zone(Zone_choisie)
all_events = []

for agenda in agendas:
    events = fetch_openagenda_events(
        agenda_uid = str(agenda["uid"]),
        zone = Zone_choisie,
        scope = scope_choisi
    )
    
    all_events.extend(events)

In [12]:
    
print(f"Nombre d'événements récupérés : {len(all_events)}")
print(all_events[:1])

Nombre d'événements récupérés : 196
[{'uid': 44260200, 'slug': 'patronage-de-la-maisonnee-saint-joseph', 'canonicalUrl': 'https://openagenda.com/saint-jean-saint-joseph-montpellier/events/patronage-de-la-maisonnee-saint-joseph', 'title': {'fr': 'Patronage de la Maisonnée Saint-Joseph'}, 'description': {'fr': 'Avec le patronage de la Maisonnée Saint-Joseph'}, 'longDescription': {'fr': '**La Maisonnée Saint Joseph est un patronage dont la devise est "Ici on joue, ici on prie, ici on s\'aime" ouvert dans un esprit évangélique aux garçons de 6 à 21 ans.**\n\n### Programme 2025-2026\n\n6 Septembre\xa0: ouverture de la maisonnée  \n10 Septembre ouverture de la petite œuvre  \n13-14 Septembre Premier week-end de rassemblement  \n11-12 Octobre Deuxième week-end de rassemblement  \n11 Octobre Réunion avec les parents 17h30  \n15-16 Novembre Troisième week-end de rassemblement  \n13-14 Décembre Quatrième week-end de rassemblement  \n17-18 Janvier Cinquième week-end de rassemblement  \n14-15 Févr

In [13]:
#transforme json
df_events = normalize_events(all_events)
#cré colonne recap
df_events = build_indexable_text(df_events)

print(df_events.shape)
print(df_events.head(3))

(196, 12)
  event_uid agenda_uid                                          title  \
0  44260200                    Patronage de la Maisonnée Saint-Joseph   
1  32761013                                           Maison de Marie   
2  66667569             Messe des papas et des grands de la Maisonnée   

                                      description  \
0  Avec le patronage de la Maisonnée Saint-Joseph   
1    Œuvre d’Education de la Paroisse St Cléophas   
2                Les jeudis à 7h00 à la Maisonnée   

                        location_name         city     region  first_date  \
0  Montpellier - Prieuré Saint-Joseph  Montpellier  Occitanie  2025-10-11   
1  Montpellier - Prieuré Saint-Joseph  Montpellier  Occitanie  2025-10-11   
2  Montpellier - Prieuré Saint-Joseph  Montpellier  Occitanie  2024-09-19   

    last_date event_type                                         source_url  \
0  2026-07-24             https://openagenda.com/saint-jean-saint-joseph...   
1  2026-07-20    

In [14]:
#finalise en transformant le document selon schama imposé 
documents = [
    build_event_document(row.to_dict())
    for _, row in df_events.iterrows()
]

In [15]:
display(documents)

[Document(metadata={'doc_id': 'openagenda_44260200', 'chunk_id': 'openagenda_44260200_0', 'source': 'openagenda', 'event_uid': '44260200', 'agenda_uid': '', 'title': 'Patronage de la Maisonnée Saint-Joseph', 'city': 'Montpellier', 'region': 'Occitanie', 'location_name': 'Montpellier - Prieuré Saint-Joseph', 'first_date': '2025-10-11', 'last_date': '2026-07-24', 'event_type': '', 'url': 'https://openagenda.com/saint-jean-saint-joseph-montpellier/events/patronage-de-la-maisonnee-saint-joseph', 'lang': 'fr'}, page_content='Titre : Patronage de la Maisonnée Saint-Joseph\nDescription : Avec le patronage de la Maisonnée Saint-Joseph\nLieu : Montpellier - Prieuré Saint-Joseph\nVille : Montpellier\nRégion : Occitanie\nDate de début : 2025-10-11\nDate de fin : 2026-07-24'),
 Document(metadata={'doc_id': 'openagenda_32761013', 'chunk_id': 'openagenda_32761013_0', 'source': 'openagenda', 'event_uid': '32761013', 'agenda_uid': '', 'title': 'Maison de Marie', 'city': 'Montpellier', 'region': 'Occit

In [16]:
#embedding
embeddings = MistralAIEmbeddings(
    model="mistral-embed"
)

#vectorisation
vectorstore = FAISS.from_documents(
    documents,
    embeddings
)

In [17]:
#RECHERCHE simulée
print("Nombre de documents indexés :", vectorstore.index.ntotal)
print("Dimension des vecteurs :", vectorstore.index.d)

query = "exposition architecture"

results = vectorstore.similarity_search(query, k=1)

print("\nRésultats de recherche :")

for i, doc in enumerate(results, start=1):
    print(f"\n--- Résultat {i} ---")
    print(doc.page_content)

Nombre de documents indexés : 196
Dimension des vecteurs : 1024

Résultats de recherche :

--- Résultat 1 ---
Titre : Exposition : « Dessiner l'Architecture #1 »
Description : Aquarelles, relevés, croquis d’étude, esquisses, perspectives, dessins techniques, rendus de concours...
Lieu : En Traits Libres
Ville : Montpellier
Région : Occitanie
Date de début : 2025-09-18
Date de fin : 2025-09-18


In [18]:
#verification de l'index objet vectorstore
print("Type d'index :", type(vectorstore.index))
print("Nombre de vecteurs indexés :", vectorstore.index.ntotal)
print("Dimension des vecteurs :", vectorstore.index.d)

Type d'index : <class 'faiss.swigfaiss_avx2.IndexFlatL2'>
Nombre de vecteurs indexés : 196
Dimension des vecteurs : 1024


In [19]:
#verification tous  les documents sont indexés
print("Nombre de documents source :", len(documents))
print("Nombre de vecteurs dans l'index :", vectorstore.index.ntotal)

assert vectorstore.index.ntotal == len(documents), \
    "Attention : tous les documents n'ont pas été indexés"

Nombre de documents source : 196
Nombre de vecteurs dans l'index : 196


In [20]:
#mon retriever avec k=1
retriever = vectorstore.as_retriever(search_kwargs={"k": 1})

In [21]:
# LLM
llm = ChatMistralAI(
    model="mistral-small-latest",
    temperature=0.2
)

prompt = ChatPromptTemplate.from_template("""
Tu es un assistant spécialisé dans la recommandation d'événements culturels.

Ton objectif est d'aider l'utilisateur à découvrir des événements intéressants à partir des informations disponibles.

Utilise uniquement les événements présents dans le contexte fourni. 
Ne crée jamais d'événement qui n'apparaît pas dans ce contexte. 
Si aucun événement ne correspond à la demande de l'utilisateur, indique-le simplement et propose éventuellement de reformuler la recherche.

Lorsque tu proposes un événement, présente-le de manière claire et naturelle en mentionnant :
- le titre de l'événement
- le lieu
- la ville
- la date
- une courte explication indiquant pourquoi cet événement correspond à la demande de l'utilisateur.

Rédige ta réponse de façon fluide, comme si tu conseillais quelqu'un qui cherche une idée de sortie culturelle.
                                          
Question utilisateur :
{question}

Contexte :
{context}
""")

In [22]:
def format_docs(docs):
    return "\n\n".join(
        f"Titre: {d.metadata.get('title', '')}\n"
        f"Lieu: {d.metadata.get('location_name', '')}\n"
        f"Ville: {d.metadata.get('city', '')}\n"
        f"Date début: {d.metadata.get('first_date', '')}\n"
        f"Date fin: {d.metadata.get('last_date', '')}\n"
        f"Description: {d.page_content}"
        for d in docs
    )

In [23]:
def chatbot(question: str) -> str:
    docs = retriever.invoke(question)
    context = format_docs(docs)
    chain = prompt | llm | StrOutputParser()
    return chain.invoke({
        "question": question,
        "context": context
    })

In [24]:
response = chatbot("Je cherche une exposition d'architecture à Montpellier")
print(response)

Voici une exposition qui correspond parfaitement à votre recherche :

**« Dessiner l'Architecture #1 »**
📍 **En Traits Libres** – Montpellier
📅 **Le 18 septembre 2025**

Cette exposition met en avant des œuvres graphiques variées autour de l'architecture : aquarelles, croquis d'étude, perspectives et dessins techniques. Une belle occasion de découvrir comment les architectes et artistes capturent l'espace et la forme à travers le dessin.
